# Kvantificering af fælles designarv i en effekttransformerflåde med PROC INBREED

## Resumé

En netoperatørs stationstransformere konstrueres i successive designgenerationer, hvor hver ny model er udledt af to forgængerdesigns. Dette notebook behandler denne konstruktionsmæssige slægtshistorie som en stamtavle og bruger **PROC INBREED** til at beregne indavls- og additive slægtskabs- (kovarians-) koefficienter, der kvantificerer, hvor meget designarv to aktiver deler.

Stamtavlen er opbygget, så strukturen er let at fortolke: to af de fire nuværende flådemodeller nedstammer fra et par **søskende**-forgængerdesigns og bærer derfor en koncentreret arv, mens de øvrige trækker på adskilte slægtslinjer. Proceduren genfinder dette præcist. De to søskende-udledte modeller (`G3_T1`, `G3_T3`) har hver en indavlskoefficient på **F = 0,25**; de to øvrige (`G3_T2`, `G3_T4`) har **F = 0**. Flådens gennemsnitlige indavlskoefficient er **0,0417**. Ser man på den additive slægtskabsmatrix, er det mindst beslægtede par i den nuværende flåde **`G3_T1` og `G3_T3` (slægtskab = 0)** — de deler ingen aner og udgør den stærkeste redundante (N-1) parring, hvilket er vigtigt, fordi `G3_T1` selv er et af de mest indavlede designs.

## Datakilder

| Datasæt | Rækker | Nøglevariable | Beskrivelse |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | En lille, deterministisk konstruktionsstamtavle for stationstransformerdesigns på tværs af tre ikke-overlappende designgenerationer (4 grundlæggende platforme, 4 anden-generations afledninger, 4 nuværende flådemodeller). Hvert ikke-grundlæggende design registrerer de to distinkte forgængerdesigns (`Parent1`, `Parent2`), det er udledt fra. `Sex` markerer den ledende designlinje (M = mekanisk/kerne-linje, F = elektrisk/vikling-linje). Stamtavlen er håndspecificeret — ikke tilfældig — så slægtskabsstrukturen er kendt på forhånd og kan kontrolleres mod procedurens output. |

# Kvantificering af fælles designarv i en effekttransformerflåde

## Hvorfor et forsyningsselskab bekymrer sig om "indavl"

Et transmissions- og distributionsselskab driver hundredvis af stationstransformere. For at styre konstruktionsomkostninger og kvalificeringsindsats designer producenter sjældent hver transformer fra bunden — hver ny model **arver** kernegeometri, viklingstopologi, isoleringssystemer og gennemføringsdesigns fra én eller to forgængermodeller. Over flere indkøbscyklusser skaber dette en ægte *konstruktionsmæssig slægtshistorie*: en stamtavle af designs.

Denne fælles arv er en skjult driftssikkerhedsrisiko. Hvis to transformere, der beskytter samme last (et redundant **N-1**-par), nedstammer fra samme oprindelige design, vil en latent designfejl — en resonanstilstand, en isoleringsaldringsmekanisme, en gennemføringsoverslagsvej — sandsynligvis være til stede i **begge** enheder. Én grundlæggende årsag kan dermed slå det redundante par ud samtidigt: et *fælles fejlmodus* (common-mode failure).

**PROC INBREED** blev bygget til netop at kvantificere denne form for fælles herkomst. Selvom dens oprindelse ligger i dyre- og planteavl, er dens matematik — Wrights additive slægtskabsrekursion — domæneuafhængig: den måler den andel af designarv, to individer deler gennem fælles aner. Vi overfører genetikkens begreber til aktivkonstruktion:

| INBREED-begreb | Fortolkning for forsyningsselskabet |
|---|---|
| Individ | Et transformerdesign / en model |
| Forælder (han/hun) | Et forgængerdesign, det er udledt fra |
| Generation (CLASS) | En indkøbs-/designcyklus |
| Indavlskoefficient *F* | Grad af selv-lignende arv inden for et design |
| Kovarians-/slægtskabskoefficient | Fælles designarv mellem to designs |
| Mindst beslægtede par | Bedste redundante parring for N-1-robusthed |

## Trin 1 — Angiv designstamtavlen

Vi indtaster `DesignLineage` direkte, så slægtskabsstrukturen er entydig. Der er tre **ikke-overlappende designgenerationer**:

- **Generation 1** — fire grundlæggende platformdesigns (`G1_P1`-`G1_P4`) uden forgængere (tomme forældre).
- **Generation 2** — fire afledte designs (`G2_D1`-`G2_D4`), hver konstrueret ud fra to **distinkte** generation-1-platforme. `G2_D1` og `G2_D2` er *fulde søskende* (begge fra `G1_P1` x `G1_P2`); `G2_D3` og `G2_D4` er fulde søskende (begge fra `G1_P3` x `G1_P4`).
- **Generation 3** — fire nuværende flådemodeller (`G3_T1`-`G3_T4`). `G3_T1` er bygget fra søskendeparret `G2_D1` x `G2_D2`, og `G3_T3` fra søskendeparret `G2_D3` x `G2_D4`; disse to designs koncentrerer derfor arven. `G3_T2` og `G3_T4` krydser hver de to adskilte slægtslinjer.

Kolonnerne `ID`, `Parent1` og `Parent2` bærer stamtavlen; `Sex` registrerer, hvilken konstruktionslinje der ledte designet. Et kort opfølgende DATA-trin tømmer placeholder-punktummet `.`, så de grundlæggende platforme har tomme forældre, som PROC INBREED forventer.

In [1]:
data DesignLineage;
   LÆNGDE ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   INDDATA Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
KØR;

/* Grundlæggende platforme har ingen forgængere: tøm placeholder-punktummerne */
data DesignLineage;
   SÆT DesignLineage;
   HVIS Parent1 = '.' SÅ Parent1 = ' ';
   HVIS Parent2 = '.' SÅ Parent2 = ' ';
KØR;

TITEL 'Transformerdesign-stamtavle';
PROCEDURE UDSKRIV data=DesignLineage noobs;
   VARIABEL Generation ID Parent1 Parent2 Sex;
   MÆRKAT Generation='Generation' ID='Design-id' Parent1='Forgænger 1' Parent2='Forgænger 2' Sex='Ledende linje';
KØR;

                                              Transformerdesign-stamtavle                                               


Generation  Design-id   Forgænger 1   Forgænger 2  Ledende linje
----------  ---------  ------------  ------------  -------------
         1  G1_P1                                  M
         1  G1_P2                                  M
         1  G1_P3                                  M
         1  G1_P4                                  F
         2  G2_D1      G1_P1         G1_P2         M
         2  G2_D2      G1_P1         G1_P2         F
         2  G2_D3      G1_P3         G1_P4         F
         2  G2_D4      G1_P3         G1_P4         M
         3  G3_T1      G2_D1         G2_D2         M
         3  G3_T2      G2_D1         G2_D3         F
         3  G3_T3      G2_D3         G2_D4         F
         3  G3_T4      G2_D2         G2_D4         M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Transformerdesign-stamtavle.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Trin 2 — Indavlskoefficienter efter designgeneration

Vi kører PROC INBREED i **flergenerations-tilstand** ved at angive `Generation` i `CLASS`-sætningen, hvilket udskriver stamtavlesammendraget efter generation. `VAR`-sætningen angiver de tre stamtavlekolonner i den krævede rækkefølge: individ, første forgænger, anden forgænger.

- **IND**-tilvalget udskriver hvert designs indavlskoefficient *F* — et mål for, hvor koncentreret dets egen arv er. Et design bygget ud fra to nært beslægtede forgængere har en høj *F*.
- **MATRIX**-tilvalget udskriver den fulde additive slægtskabsmatrix, så vi kan aflæse parvis arv direkte.
- **AVERAGE**-tilvalget tilføjer flådens gennemsnitlige indavlskoefficient — et samlet designmangfoldighedsmål.

Værdier nær 0 betyder uafhængige slægtslinjer; *F* stiger, efterhånden som et designs to forgængere bliver tættere beslægtede.

In [2]:
TITEL 'Indavlskoefficienter efter designgeneration';
PROCEDURE inbreed data=DesignLineage ind average MATRIX;
   VARIABEL ID Parent1 Parent2;
   KLASSE Generation;
   MÆRKAT ID='Design-id' Parent1='Forgænger 1' Parent2='Forgænger 2' Generation='Generation';
KØR;

                                      Indavlskoefficienter efter designgeneration                                       


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
Design-id           F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
Design-id           F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Indavlskoefficienter efter designgeneration.
NOTE: PROC INBREED data=DesignLineage



## Trin 3 — Kovarianskoefficienter gemt til efterfølgende risikoscoring

Ved at erstatte indavlsvisningen med **COVAR**-tilvalget rapporteres de samme relationer som **kovarians- (additive slægtskabs-) koefficienter**, den form et aktivstyringsteam ville anvende i en redundans-risikoscore. **OUTCOV=**-tilvalget gemmer den fulde matrix som et datasæt (`DesignCovar`), hvor kolonnerne `Col1`-`Col12` indeholder hvert designs slægtskab til alle andre designs (i stamtavlerækkefølge) — klar til at kobles til stationstildelinger.

Vi udskriver outputdatasættet, så den gemte matrix er synlig sammen med listen.

In [3]:
TITEL 'Kovarians- (slægtskabs-) koefficienter';
PROCEDURE inbreed data=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   VARIABEL ID Parent1 Parent2;
   KLASSE Generation;
   MÆRKAT ID='Design-id' Parent1='Forgænger 1' Parent2='Forgænger 2' Generation='Generation';
KØR;

TITEL 'OUTCOV= slægtskabsmatrix gemt til risikoscoring';
PROCEDURE UDSKRIV data=DesignCovar noobs;
KØR;

                                         Kovarians- (slægtskabs-) koefficienter                                         


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
Design-id           F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
Design-id              G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Kovarians- (slægtskabs-) koefficienter.
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to OUTCOV= slægtskabsmatrix gemt til risikoscoring.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Trin 4 — Mindst beslægtede parringer til redundante (N-1) installationer

Den gemte `DesignCovar`-matrix er præcis, hvad et redundansstudie har brug for. Slægtskabet mellem design *i* og design *j* findes i række *i*, kolonne `Col`*j* (kolonnerne er i stamtavlerækkefølge, så `Col9`-`Col12` svarer til de fire nuværende flådemodeller `G3_T1`-`G3_T4`).

Vi henter de fire nuværende flådemodellers rækker fra `DesignCovar`, danner hvert uordnet par af dem, tilføjer deres slægtskabskoefficient og sorterer mindst beslægtede først. Målet er at vælge redundante par, hvis fælles arv er **lavest** — det minimerer chancen for, at én arvet designfejl sætter begge halvdele af en N-1-beskyttet position ud af drift.

In [4]:
/* Hent de fire nuværende flådemodellers rækker; Col9..Col12 er slægtskaberne
   til G3_T1..G3_T4 (stamtavlens kolonnerækkefølge). Byg hvert uordnet par. */
data Pairs;
   SÆT DesignCovar;
   HVOR ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   LÆNGDE DesignA $12 DesignB $12;
   TABEL g3{4} Col9 Col10 Col11 Col12;
   TABEL nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   GØR j = 1 TIL 4;
      DesignB = nm{j};
      HVIS DesignA < DesignB SÅ GØR;
         Relationship = g3{j};
         UDDATA;
      SLUT;
   SLUT;
   BEHOLD DesignA DesignB Relationship;
KØR;

PROCEDURE SORTER data=Pairs; EFTER Relationship; KØR;

TITEL 'Kandidat-redundante (N-1) parringer, mindst beslægtede først';
PROCEDURE UDSKRIV data=Pairs noobs;
   VARIABEL DesignA DesignB Relationship;
   MÆRKAT DesignA='Design A' DesignB='Design B' Relationship='Slægtskab';
KØR;
TITEL;

                              Kandidat-redundante (N-1) parringer, mindst beslægtede først                              


Design A  Design B   Slægtskab
--------  --------  ----------
G3_T1     G3_T3              0
G3_T2     G3_T4           0.25
G3_T1     G3_T2          0.375
G3_T1     G3_T4          0.375
G3_T2     G3_T3          0.375
G3_T3     G3_T4          0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Kandidat-redundante (N-1) parringer, mindst beslægtede først.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Fortolkning af resultaterne

**Indavlskoefficienter (Trin 2).** Alle grundlæggende generation-1-platforme og alle generation-2-afledninger viser *F* = 0 — per konstruktion har ingen af dem to beslægtede forgængere. To generation-3-modeller fremstår med *F* = 0,25: `G3_T1` (bygget fra søskendeparret `G2_D1` x `G2_D2`) og `G3_T3` (fra søskendeparret `G2_D3` x `G2_D4`). Deres forgængere kan spores tilbage til *samme* par af grundlæggende platforme, så deres arv er koncentreret. Set fra et driftssikkerhedsperspektiv er disse de designs, der er mest udsatte for en enkelt arvet fejl, og de berettiger ekstra uafhængig kvalificeringstest. `G3_T2` og `G3_T4`, som krydser de to adskilte slægtslinjer, har *F* = 0.

**Slægtskabsmatrix (Trin 3).** Elementerne uden for diagonalen kvantificerer parvis fælles arv. De to søskende-generation-2-par viser hver et slægtskab på 0,50 til hinanden (`G2_D1`-`G2_D2` og `G2_D3`-`G2_D4`), mens afledninger fra modsatte slægtslinjer ligger på 0,00. De indavlede nuværende flådedesigns har selvslægtskaber på 1,25 (= 1 + *F*), synligt på diagonalen. `DesignCovar`-datasættet gør den fulde matrix tilgængelig til at koble mod stationstildelinger.

**Mindst beslægtede parringer (Trin 4).** Rangordnes hvert par i den nuværende flåde efter slægtskab, placerer det **`G3_T1` og `G3_T3` først med slægtskab 0,00** — de nedstammer fra helt adskilte oprindelige platforme og deler ingen designarv. Dette er den stærkeste redundante parring, og den er særligt værdifuld, fordi `G3_T1` selv er et af de to mest indavlede designs: at parre det med en helt ubeslægtet modpart afbøder dets koncentrerede arv. Det næstbedste par er `G3_T2` og `G3_T4` med 0,25; de resterende par ligger på 0,375. Flådens gennemsnitlige indavlskoefficient på **0,0417** (udskrevet af AVERAGE-tilvalget i Trin 2) opsummerer den samlede designmangfoldighed. Indkøb bør foretrække parrene med lavest slægtskab til N-1-positioner og over tid udvide det oprindelige grundlag — aktivkonstruktionens modstykke til at undgå en genetisk flaskehals.

**Forbehold.** Dette er illustrative syntetiske data; et produktionsstudie ville opbygge stamtavlen ud fra producentens reelle designrevisionsregistreringer og validere slægtskabsscorerne mod historiske fælles fejlmodus-driftsstop, før det driver indkøbsbeslutninger.